In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataloader import Loader
import warnings
import os
import time
import math
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)



In [6]:
class TinyEEGCNN(nn.Module):
    def __init__(self, in_ch=41):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),  # -> [B, 128, 1]
        )
        self.fc = nn.Linear(128, 1)  # binary logit

    def forward(self, x):
        h = self.net(x).squeeze(-1)  # [B, 128]
        return self.fc(h).squeeze(-1)  # [B]


d=Loader(batch_size=128)
dataloader=d.return_Loader()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyEEGCNN().to(device)
print("Model device:", next(model.parameters()).device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()


print(device)

Model device: cuda:0
cuda


In [7]:
writer = SummaryWriter("runs/tinycnn") 
epochs = 30
step = 0
for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total = 0
        correct = 0
        all_targets = []
        all_probs = []

        for batch in dataloader:
            x = batch["x"].to(device, non_blocking=True).float()
            y = batch["y"].to(device, non_blocking=True).float()
            y_flat = y.view(-1)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            logits_flat = logits.view(-1)
            loss = criterion(logits, y)
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            total += x.size(0)

            with torch.no_grad():
                probs= torch.sigmoid(logits)
                preds = (probs >= 0.5).float()
                correct += (preds == y).sum().item()
                all_targets.append(y_flat.detach().cpu())
                all_probs.append(probs.detach().cpu())

            writer.add_scalar("train/loss_step", loss.item(), step)
            step += 1

        epoch_loss = total_loss / max(total, 1)
        epoch_acc = correct / max(total, 1)
        y_true = torch.cat(all_targets).numpy().astype(int)
        y_prob = torch.cat(all_probs).numpy()
        y_pred = (y_prob >= 0.5).astype(int)

        # confusion matrix -> tn, fp, fn, tp
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        specificity = (tn / (tn + fp)) if (tn + fp) > 0 else 0.0

        # AUC metrics can fail if only one class appears in y_true for the epoch
        if len(set(y_true.tolist())) == 2:
            roc_auc = roc_auc_score(y_true, y_prob)
            pr_auc = average_precision_score(y_true, y_prob)
        else:
            roc_auc = float("nan")
            pr_auc = float("nan")

        # ----- log epoch scalars -----
        writer.add_scalar("train/loss_epoch", epoch_loss, epoch)
        writer.add_scalar("train/acc_epoch", epoch_acc, epoch)

        writer.add_scalar("train/precision", precision, epoch)
        writer.add_scalar("train/recall", recall, epoch)
        writer.add_scalar("train/f1", f1, epoch)
        writer.add_scalar("train/specificity", specificity, epoch)

        writer.add_scalar("train/roc_auc", roc_auc, epoch)
        writer.add_scalar("train/pr_auc", pr_auc, epoch)

        # optional: log raw confusion counts (super useful)
        writer.add_scalar("train/TP", tp, epoch)
        writer.add_scalar("train/FP", fp, epoch)
        writer.add_scalar("train/TN", tn, epoch)
        writer.add_scalar("train/FN", fn, epoch)

        print(
            f"Epoch {epoch:02d} | loss={epoch_loss:.4f} | acc={epoch_acc:.3f} | "
            f"P={precision:.3f} R={recall:.3f} F1={f1:.3f} Spec={specificity:.3f} | "
            f"ROC-AUC={roc_auc:.3f} PR-AUC={pr_auc:.3f}"
        )

writer.close()
print("Done. Run: tensorboard --logdir runs")

Epoch 01 | loss=0.6424 | acc=0.710 | P=0.234 R=0.004 F1=0.008 Spec=0.995 | ROC-AUC=0.487 PR-AUC=0.283
Epoch 02 | loss=0.6079 | acc=0.714 | P=0.522 R=0.023 F1=0.043 Spec=0.992 | ROC-AUC=0.542 PR-AUC=0.328
Epoch 03 | loss=0.5843 | acc=0.715 | P=0.547 R=0.044 F1=0.081 Spec=0.985 | ROC-AUC=0.629 PR-AUC=0.390
Epoch 04 | loss=0.5588 | acc=0.730 | P=0.609 R=0.164 F1=0.259 Spec=0.958 | ROC-AUC=0.690 PR-AUC=0.467
Epoch 05 | loss=0.5423 | acc=0.740 | P=0.640 R=0.213 F1=0.319 Spec=0.952 | ROC-AUC=0.718 PR-AUC=0.509
Epoch 06 | loss=0.5276 | acc=0.747 | P=0.655 R=0.248 F1=0.360 Spec=0.947 | ROC-AUC=0.740 PR-AUC=0.543
Epoch 07 | loss=0.5182 | acc=0.750 | P=0.658 R=0.270 F1=0.383 Spec=0.944 | ROC-AUC=0.755 PR-AUC=0.561
Epoch 08 | loss=0.5080 | acc=0.757 | P=0.673 R=0.300 F1=0.415 Spec=0.941 | ROC-AUC=0.767 PR-AUC=0.582
Epoch 09 | loss=0.4988 | acc=0.767 | P=0.694 R=0.339 F1=0.455 Spec=0.940 | ROC-AUC=0.778 PR-AUC=0.601
Epoch 10 | loss=0.4898 | acc=0.775 | P=0.709 R=0.368 F1=0.485 Spec=0.939 | ROC-AUC

KeyboardInterrupt: 

In [8]:

model.eval()


TinyEEGCNN(
  (net): Sequential(
    (0): Conv1d(41, 64, kernel_size=(7,), stride=(2,), padding=(3,))
    (1): ReLU()
    (2): Conv1d(64, 128, kernel_size=(5,), stride=(2,), padding=(2,))
    (3): ReLU()
    (4): AdaptiveAvgPool1d(output_size=1)
  )
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [14]:
import torch
from collections import defaultdict

test_loader=Loader('cache_windows_eval/manifest.jsonl',transform=None,
        batch_size=32,
        shuffle=False, # it's the issue
        num_workers=2,
        pin_memory=False,)
test_loader=test_loader.return_Loader()


In [15]:
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score

@torch.no_grad()
def evaluate_1d(model, loader, device, thr=0.5):
    model.eval()

    all_probs = []
    all_y = []

    for batch in loader:
        # adapt these keys if your Loader is different
        x = batch["x"].to(device).float()     # [B, C, T]
        y = batch["y"].to(device).float()     # [B]

        logits = model(x)                     # [B]
        probs = torch.sigmoid(logits)         # [B]

        all_probs.append(probs.cpu())
        all_y.append(y.cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_y = torch.cat(all_y).numpy()

    # Metrics
    auc = roc_auc_score(all_y, all_probs) if len(np.unique(all_y)) > 1 else float("nan")
    pred = (all_probs >= thr).astype(np.int32)

    acc = accuracy_score(all_y, pred)
    pre, rec, f1, _ = precision_recall_fscore_support(all_y, pred, average="binary", zero_division=0)

    return {
        "acc": float(acc),
        "pre": float(pre),
        "rec": float(rec),
        "f1": float(f1),
        "auc": float(auc),
    }


In [16]:
metrics = evaluate_1d(model, test_loader, device, thr=0.5)
print(metrics)


{'acc': 0.8007152465632075, 'pre': 0.52405834966253, 'rec': 0.3823669579030977, 'f1': 0.44213813372520205, 'auc': 0.7304338468616434}


In [17]:
torch.save(model.state_dict(), 'model_weights_tiny1d.pth')